
**Meenakshi Gopakumar Nair — Undergraduate Project**

This notebook implements the part 1 pipeline:
1. YOLOv8 person detection
2. Grid-based density estimation
3. Optical flow motion analysis
4. Risk classification & alert generation
5. Visualisation

---
**Before running**: Go to **Runtime → Change runtime type → L4 GPU**

## Install Dependencies

In [ ]:
!pip install ultralytics --quiet
!pip install opencv-python-headless --quiet
!pip install matplotlib numpy Pillow --quiet

print('All dependencies installed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
# First, confirm the zip is visible
print(os.listdir('/content/drive/MyDrive'))

In [ ]:
!unzip "/content/drive/MyDrive/VisDrone2019-DET-val.zip" -d "/content/visdrone"

In [ ]:
import os

# Check what's inside /content/visdrone
for root, dirs, files in os.walk('/content/visdrone'):
    print(root)
    if files:
        print('  files:', files[:3])  # show first 3 files in each folder
    break  # just show top level first

In [ ]:
import os
image_folder = '/content/visdrone/VisDrone2019-DET-val/images'  # or just '/content/visdrone' depending on how it unzips
images = os.listdir(image_folder)
print(f'Found {len(images)} images')
print('Sample:', images[:3])

# Set MEDIA_FILE to first image
MEDIA_FILE = os.path.join(image_folder, images[0])
IS_VIDEO = False
print(f'Using: {MEDIA_FILE}')

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

image_folder = '/content/visdrone/VisDrone2019-DET-val/images'
images = sorted(os.listdir(image_folder))

# Show first 12 images as a grid so you can pick one you want to use right now
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for i, ax in enumerate(axes.flat):
    img = cv2.imread(os.path.join(image_folder, images[i]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f'[{i}] {images[i][:20]}', fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
MEDIA_FILE = os.path.join(image_folder, images[3])  # change 4 to which one you pick
IS_VIDEO = False
print(f'Using: {MEDIA_FILE}')

## YOLOv8 Person Detection

load YOLOv8 medium and run detection. `classes=[0]` filters to **people only**.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

print('Loading YOLOv8 model...')
model = YOLO('yolov8m.pt')
print('Model loaded.')

In [ ]:
def detect_people(image_path, model, conf_threshold=0.25):
    results = model(image_path, classes=[0], conf=conf_threshold, verbose=False)
    result = results[0]
    boxes = result.boxes.xyxy.cpu().numpy()
    confidences = result.boxes.conf.cpu().numpy()
    annotated = result.plot()
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    return annotated_rgb, boxes, confidences

if not IS_VIDEO:
    annotated_img, boxes, confs = detect_people(MEDIA_FILE, model)
    print(f'People detected: {len(boxes)}')
    plt.figure(figsize=(12, 8))
    plt.imshow(annotated_img)
    plt.title(f'YOLOv8 Detection — {len(boxes)} people found', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    cap = cv2.VideoCapture(MEDIA_FILE)
    ret, first_frame = cap.read()
    cap.release()
    if ret:
        cv2.imwrite('first_frame.jpg', first_frame)
        annotated_img, boxes, confs = detect_people('first_frame.jpg', model)
        print(f'People in first frame: {len(boxes)}')
        plt.figure(figsize=(12, 8))
        plt.imshow(annotated_img)
        plt.title(f'First Frame — {len(boxes)} people', fontsize=14)
        plt.axis('off')
        plt.show()

## Grid Based Density Estimation

Divide the image into a grid and compute **people/m²** per cell using the drone's altitude and camera FOV.

| Density | Risk Level |
|---|---|
| < 2 /m² | Safe |
| 2–4 /m² | Caution |
| 4–5 /m² | Warning |
| 5–7 /m² | Danger |
| > 7 /m² | Critical |

In [ ]:
import math

GRID_ROWS = 4
GRID_COLS = 4
DRONE_ALTITUDE_M = 30
CAMERA_FOV_DEG = 84

RISK_LEVELS = [
    (7.0, 'CRITICAL',  '#FF0000'),
    (5.0, 'DANGER',    '#FF6600'),
    (4.0, 'WARNING',   '#FFCC00'),
    (2.0, 'CAUTION',   '#99CC00'),
    (0.0, 'SAFE',      '#00AA44'),
]

RISK_ORDER = ['SAFE', 'CAUTION', 'WARNING', 'DANGER', 'CRITICAL']

RISK_COLOURS = {
    'SAFE':     '#00AA44',
    'CAUTION':  '#99CC00',
    'WARNING':  '#FFCC00',
    'DANGER':   '#FF6600',
    'CRITICAL': '#FF0000',
}

def density_to_risk(density):
    for threshold, label, colour in RISK_LEVELS:
        if density >= threshold:
            return label, colour
    return 'SAFE', '#00AA44'

def compute_density_grid(boxes, image_shape,
                         grid_rows=GRID_ROWS, grid_cols=GRID_COLS,
                         drone_alt=DRONE_ALTITUDE_M, fov_deg=CAMERA_FOV_DEG):
    h, w = image_shape[:2]
    ground_width  = 2 * math.tan(math.radians(fov_deg / 2)) * drone_alt
    ground_height = ground_width * (h / w)
    cell_area     = (ground_width / grid_cols) * (ground_height / grid_rows)
    count_grid    = np.zeros((grid_rows, grid_cols), dtype=int)
    for box in boxes:
        cx  = (box[0] + box[2]) / 2
        cy  = (box[1] + box[3]) / 2
        col = min(int(cx / w * grid_cols), grid_cols - 1)
        row = min(int(cy / h * grid_rows), grid_rows - 1)
        count_grid[row][col] += 1
    density_grid = count_grid / cell_area
    return density_grid, count_grid, cell_area

def visualise_density_grid(image_path, boxes, density_grid,
                            grid_rows=GRID_ROWS, grid_cols=GRID_COLS):
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    h, w   = img_rgb.shape[:2]
    cell_h = h // grid_rows
    cell_w = w // grid_cols
    overlay = img_rgb.copy()
    for r in range(grid_rows):
        for c in range(grid_cols):
            density = density_grid[r][c]
            label, hex_colour = density_to_risk(density)
            hx  = hex_colour.lstrip('#')
            rgb = tuple(int(hx[i:i+2], 16) for i in (0, 2, 4))
            x1, y1 = c * cell_w, r * cell_h
            x2, y2 = x1 + cell_w, y1 + cell_h
            cv2.rectangle(overlay, (x1, y1), (x2, y2), rgb, -1)
            cv2.putText(img_rgb, label,         (x1+5, y1+22), cv2.FONT_HERSHEY_SIMPLEX, 0.45, rgb, 2)
            cv2.putText(img_rgb, f'{density:.1f}/m2', (x1+5, y1+42), cv2.FONT_HERSHEY_SIMPLEX, 0.4,  rgb, 1)
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), rgb, 2)
    result = cv2.addWeighted(overlay, 0.25, img_rgb, 0.75, 0)
    return result

print(' Density functions defined.')

In [ ]:
img_for_density = MEDIA_FILE if not IS_VIDEO else 'first_frame.jpg'
img_bgr = cv2.imread(img_for_density)
_, boxes, _ = detect_people(img_for_density, model)

density_grid_result, count_grid, cell_area = compute_density_grid(boxes, img_bgr.shape)

print(f'Ground area per cell: {cell_area:.2f} m²')
print('People count per cell:')
print(count_grid)
print('Density (people/m²) per cell:')
print(np.round(density_grid_result, 2))

density_vis = visualise_density_grid(img_for_density, boxes, density_grid_result)
plt.figure(figsize=(13, 9))
plt.imshow(density_vis)
plt.title('Density Grid — Colour-coded by Risk Level', fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

## Optical Flow & Motion Analysis

Optical flow tracks how pixels move between frames. A high **disorder score** (std of flow angles) means chaotic and unpredictable movement.

In [ ]:
def compute_optical_flow(frame1, frame2):
    gray1 = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    flow  = cv2.calcOpticalFlowFarneback(
        gray1, gray2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    magnitude, angle = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    disorder  = float(np.std(angle))
    avg_speed = float(np.mean(magnitude))
    hsv = np.zeros_like(frame1)
    hsv[..., 1] = 255
    hsv[..., 0] = angle * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(magnitude, None, 0, 255, cv2.NORM_MINMAX)
    flow_vis = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return flow, magnitude, angle, disorder, avg_speed, flow_vis

def motion_risk_label(disorder, avg_speed):
    if disorder > 1.8 and avg_speed > 3.0:
        return 'HIGH CHAOS',      '#FF0000'
    elif disorder > 1.4 or avg_speed > 2.0:
        return 'MODERATE CHAOS',  '#FF6600'
    elif disorder > 1.0:
        return 'LOW MOTION',      '#FFCC00'
    else:
        return 'CALM',            '#00AA44'

print('Optical flow functions defined.')

In [ ]:
if IS_VIDEO:
    cap = cv2.VideoCapture(MEDIA_FILE)
    ret1, frame1 = cap.read()
    ret2, frame2 = cap.read()
    cap.release()
else:
    frame1 = cv2.imread(MEDIA_FILE)
    rows, cols = frame1.shape[:2]
    M = np.float32([[1, 0, np.random.uniform(-4, 4)],
                    [0, 1, np.random.uniform(-4, 4)]])
    frame2 = cv2.warpAffine(frame1, M, (cols, rows))
    noise  = np.random.randint(-15, 15, frame2.shape, dtype=np.int16)
    frame2 = np.clip(frame2.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    print('Synthetic second frame created for image input.')

flow, magnitude, angle, disorder, avg_speed, flow_vis = compute_optical_flow(frame1, frame2)
motion_label, motion_colour = motion_risk_label(disorder, avg_speed)

print(f'Disorder score : {disorder:.3f}')
print(f'Average speed  : {avg_speed:.3f} px/frame')
print(f'Motion level   : {motion_label}')

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(frame1, cv2.COLOR_BGR2RGB))
axes[0].set_title('Input Frame', fontsize=12)
axes[0].axis('off')
axes[1].imshow(flow_vis)
axes[1].set_title('Optical Flow (colour=direction, brightness=speed)', fontsize=12)
axes[1].axis('off')
plt.suptitle(f'Motion — {motion_label} | Disorder: {disorder:.2f} | Speed: {avg_speed:.2f}', fontsize=13)
plt.tight_layout()
plt.show()

## Risk Classification & Alerts

In [ ]:
def bump_risk(label, motion_label):
    if motion_label in ('HIGH CHAOS', 'MODERATE CHAOS'):
        idx = RISK_ORDER.index(label)
        return RISK_ORDER[min(idx + 1, len(RISK_ORDER) - 1)]
    return label

def generate_alerts(density_grid, motion_label,
                    grid_rows=GRID_ROWS, grid_cols=GRID_COLS):
    alerts   = []
    risk_map = []
    for r in range(grid_rows):
        row_risks = []
        for c in range(grid_cols):
            density     = density_grid[r][c]
            base_label, _ = density_to_risk(density)
            final_label = bump_risk(base_label, motion_label)
            row_risks.append(final_label)
            if RISK_ORDER.index(final_label) >= RISK_ORDER.index('WARNING'):
                alerts.append({
                    'zone':       f'Row {r+1}, Col {c+1}',
                    'density':    round(density, 2),
                    'base_risk':  base_label,
                    'motion_risk': motion_label,
                    'final_risk': final_label,
                })
        risk_map.append(row_risks)
    return alerts, risk_map

alerts, risk_map = generate_alerts(density_grid_result, motion_label)

print('=' * 55)
print('        CROWD RISK ASSESSMENT REPORT')
print('=' * 55)
print(f'  People detected : {len(boxes)}')
print(f'  Max density     : {density_grid_result.max():.2f} people/m²')
print(f'  Motion level    : {motion_label}')
print('-' * 55)
if alerts:
    print(f'  {len(alerts)} zone(s) flagged:')
    for a in alerts:
        icon = '🔴' if a['final_risk'] in ('CRITICAL','DANGER') else '🟡'
        print(f'  {icon} {a["zone"]} — {a["final_risk"]} ({a["density"]} /m²)')
else:
    print('  All zones within safe thresholds.')
print('=' * 55)

## Full Dashboard

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors

fig = plt.figure(figsize=(18, 10))
fig.patch.set_facecolor('#1a1a2e')

# Panel 1 — Detection
ax1 = fig.add_subplot(2, 3, 1)
annotated_img, _, _ = detect_people(img_for_density, model)
ax1.imshow(annotated_img)
ax1.set_title(f'Person Detection\n{len(boxes)} people', color='white', fontsize=11)
ax1.axis('off')

# Panel 2 — Density overlay
ax2 = fig.add_subplot(2, 3, 2)
ax2.imshow(density_vis)
ax2.set_title('Density Grid', color='white', fontsize=11)
ax2.axis('off')

# Panel 3 — Optical flow
ax3 = fig.add_subplot(2, 3, 3)
ax3.imshow(flow_vis)
ax3.set_title(f'Optical Flow\nDisorder: {disorder:.2f}', color='white', fontsize=11)
ax3.axis('off')

# Panel 4 — Density heatmap
ax4 = fig.add_subplot(2, 3, 4)
im = ax4.imshow(density_grid_result, cmap='RdYlGn_r', vmin=0, vmax=8, interpolation='nearest')
plt.colorbar(im, ax=ax4, label='people/m²')
for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        ax4.text(c, r, f'{density_grid_result[r,c]:.1f}',
                 ha='center', va='center', fontsize=10, color='white', fontweight='bold')
ax4.set_title('Density Heatmap', color='white', fontsize=11)
ax4.set_xticks(range(GRID_COLS))
ax4.set_yticks(range(GRID_ROWS))
ax4.set_xticklabels([f'Col {i+1}' for i in range(GRID_COLS)], color='white')
ax4.set_yticklabels([f'Row {i+1}' for i in range(GRID_ROWS)], color='white')

# Panel 5 — Risk map
ax5 = fig.add_subplot(2, 3, 5)
risk_numeric = np.array([[RISK_ORDER.index(risk_map[r][c])
                          for c in range(GRID_COLS)] for r in range(GRID_ROWS)])
cmap_r = mcolors.ListedColormap([RISK_COLOURS[r] for r in RISK_ORDER])
bounds = [-0.5, 0.5, 1.5, 2.5, 3.5, 4.5]
norm   = mcolors.BoundaryNorm(bounds, cmap_r.N)
ax5.imshow(risk_numeric, cmap=cmap_r, norm=norm, interpolation='nearest')
for r in range(GRID_ROWS):
    for c in range(GRID_COLS):
        ax5.text(c, r, risk_map[r][c],
                 ha='center', va='center', fontsize=8, color='white', fontweight='bold')
patches = [mpatches.Patch(color=RISK_COLOURS[l], label=l) for l in RISK_ORDER]
ax5.legend(handles=patches, loc='upper right', fontsize=7,
           facecolor='#1a1a2e', labelcolor='white')
ax5.set_title('Combined Risk Map', color='white', fontsize=11)
ax5.set_xticks(range(GRID_COLS))
ax5.set_yticks(range(GRID_ROWS))
ax5.set_xticklabels([f'Col {i+1}' for i in range(GRID_COLS)], color='white')
ax5.set_yticklabels([f'Row {i+1}' for i in range(GRID_ROWS)], color='white')

# Panel 6 — Alert summary
ax6 = fig.add_subplot(2, 3, 6)
ax6.set_facecolor('#0f0f23')
ax6.axis('off')
lines = [
    ('ALERT SUMMARY',              'white',        14, True),
    ('',                           'white',        10, False),
    (f'People: {len(boxes)}',      'white',        10, False),
    (f'Max density: {density_grid_result.max():.2f} /m²', 'white', 10, False),
    (f'Motion: {motion_label}',    motion_colour,  10, True),
    ('',                           'white',        10, False),
    (f'Zones flagged: {len(alerts)}', 'white',     11, True),
    ('',                           'white',        10, False),
]
for a in alerts[:4]:
    col = RISK_COLOURS.get(a['final_risk'], 'white')
    lines.append((f"{a['zone']}: {a['final_risk']}", col, 9, True))
if not alerts:
    lines.append((' All zones SAFE', '#00AA44', 11, True))

y = 0.95
for text, colour, size, bold in lines:
    ax6.text(0.5, y, text, transform=ax6.transAxes,
             ha='center', va='top', fontsize=size,
             color=colour, fontweight='bold' if bold else 'normal')
    y -= 0.09

for ax in [ax1, ax2, ax3, ax4, ax5, ax6]:
    ax.set_facecolor('#1a1a2e')

plt.suptitle('🚁 Crowd Crush & Surge Detection — Pipeline Dashboard',
             color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('pipeline_dashboard.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(' Dashboard saved as pipeline_dashboard.png')

## Next Steps

| Step | Status |
|---|---|
| YOLOv8 detection | ✅ Done |
| Grid density estimation | ✅ Done |
| Optical flow | ✅ Done |
| Risk classification & alerts | ✅ Done |
| Dashboard visualisation | ✅ Done |


1. Replace grid density with **CSRNet** for smoother heatmaps
2. Connect pipeline to a ROS2 drone node